In [19]:
import pandas as pd
import numpy as np
import logging
import gensim

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel,LdaMulticore
from gensim.utils import simple_preprocess

import spacy

from nltk.corpus import stopwords as nltk_stopwords
import nltk




In [14]:
nlp = spacy.load("fr_core_news_sm",disable = ['ner', 'tagger', 'parser', 'textcat'])
print("✓ Loaded fr_core_news_sm (lightweight, CPU-optimized)")

# Download French stopwords from NLTK
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
french_stopwords = set(nltk_stopwords.words('french'))

✓ Loaded fr_core_news_sm (lightweight, CPU-optimized)


In [3]:
# Convert to pandas DataFrame
df = pd.read_csv("/home/robin/Code_repo/psycholinguistic2125/JADT_rap_fr/data/20251125_cleaned_lrfaf_corpus.csv")
df = df[(df.year > 1991) & (df.year < 2024)]
print(f"Dataset loaded: {len(df)} verses/songs")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['year'].min()} to {df['year'].max()}")

Dataset loaded: 36498 verses/songs
Columns: ['artist', 'title', 'year', 'lyrics', 'lyrics_cleaned', 'born_in_france', 'url', 'birthdate_artist', 'age_artist', 'born_in_france.1']
Date range: 1992.0 to 2023.0


In [16]:
def preprocess_text(text, nlp_model = nlp):
    '''
    Tokenize and lemmatize text
    '''
    # Tokenize
    doc = nlp_model(text.lower())
    processed_tokens = []
    for token in doc:
        if (token.text in french_stopwords or 
            token.is_punct or 
            len(token.text) <= 2 or 
            token.is_digit):
            continue

        processed_tokens.append(token.text)
        
    return processed_tokens

# Preprocess all documents
docs = df['lyrics_cleaned'].astype(str).tolist()
processed_docs = [preprocess_text(doc) for doc in docs]
processed_docs = [doc for doc in processed_docs if len(doc) > 0]


In [27]:

print("\nCreating dictionary and corpus...")
dictionary = corpora.Dictionary(processed_docs)

dictionary.filter_extremes(no_below=20, no_above=0.6)

print(f"Dictionary size: {len(dictionary)} unique tokens")

# Create corpus (bag-of-words representation)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]


Creating dictionary and corpus...
Dictionary size: 25085 unique tokens


In [28]:
lda_model = LdaMulticore(
    corpus=corpus,
    id2word=dictionary,
    num_topics=20,
    random_state=42,
    workers= 8

)

/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/pope

In [29]:
topics = lda_model.print_topics(num_topics=-1, num_words=12)
for topic_id, topic_words in topics:
    print(f"\nTopic {topic_id}:")
    print(f"  {topic_words}")



Topic 0:
  0.008*"veux" + 0.006*"hey" + 0.006*"faut" + 0.005*"sans" + 0.005*"trop" + 0.005*"vie" + 0.005*"rap" + 0.004*"tous" + 0.004*"monde" + 0.004*"toujours" + 0.004*"fais" + 0.004*"temps"

Topic 1:
  0.007*"rien" + 0.007*"trop" + 0.007*"tous" + 0.006*"sans" + 0.006*"vie" + 0.006*"faut" + 0.005*"fais" + 0.005*"temps" + 0.004*"bien" + 0.004*"jamais" + 0.004*"-moi" + 0.004*"ouais"

Topic 2:
  0.012*"fais" + 0.006*"rien" + 0.005*"tous" + 0.004*"trop" + 0.004*"sans" + 0.004*"bien" + 0.004*"rap" + 0.004*"veut" + 0.004*"ouais" + 0.004*"dit" + 0.003*"sais" + 0.003*"vie"

Topic 3:
  0.009*"fais" + 0.008*"vie" + 0.007*"trop" + 0.006*"bien" + 0.006*"yeah" + 0.006*"qu’" + 0.005*"veux" + 0.005*"tous" + 0.005*"faut" + 0.005*"monde" + 0.004*"sais" + 0.004*"rien"

Topic 4:
  0.007*"vie" + 0.007*"trop" + 0.006*"sans" + 0.006*"fais" + 0.006*"temps" + 0.006*"-moi" + 0.005*"veux" + 0.005*"tous" + 0.005*"rien" + 0.004*"qu’" + 0.004*"aime" + 0.004*"ouais"

Topic 5:
  0.012*"ouais" + 0.008*"tous" + 0.00

In [30]:
# ============================================================
# PART 9: ANALYZE DOCUMENT-TOPIC DISTRIBUTION
# ============================================================
print("\n\n" + "="*70)
print("DOCUMENT-TOPIC DISTRIBUTION (Sample)")
print("="*70)

for doc_idx in range(min(5, len(corpus))):
    print(f"\nDocument {doc_idx}:")
    print(f"  Original text: {docs[doc_idx][:100]}...")

    doc_topics = lda_model.get_document_topics(corpus[doc_idx])
    if doc_topics:
        doc_topics_sorted = sorted(doc_topics, key=lambda x: x[1], reverse=True)

        for topic_id, prob in doc_topics_sorted:
            bar_width = int(prob * 50)
            bar = "█" * bar_width + "░" * (50 - bar_width)
            print(f"  Topic {topic_id}: {bar} {prob:.2%}")

        dominant_topic = doc_topics_sorted[0]
        print(f"  → Dominant: Topic {dominant_topic[0]} ({dominant_topic[1]:.2%})")



DOCUMENT-TOPIC DISTRIBUTION (Sample)

Document 0:
  Original text: Aucune force d'état ne peut stopper une chienne en rut
Surtout pas quand c'est la putain d'fille d'u...
  Topic 14: ███████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 23.22%
  Topic 5: ██████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 21.58%
  Topic 2: ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 12.76%
  Topic 17: █████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 10.67%
  Topic 19: █████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 10.25%
  Topic 0: █████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 10.08%
  Topic 3: ███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 7.72%
  Topic 13: █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 2.55%
  → Dominant: Topic 14 (23.22%)

Document 1:
  Original text: On vient de M.A.R.S. ce n'est pas une farce
IAM live de la planète Mars....
Eille, soleil, devient u...
  Topic 4: ██████████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 37.34%
  Topic 19: █████████░░░░░░░░░░░░░░░░░

In [31]:
print("\n\nGenerating interactive visualization...")

try:
    import pyLDAvis.gensim_models as gensimvis
    import pyLDAvis

    # Prepare visualization
    vis_data = gensimvis.prepare(lda_model, corpus, dictionary)

    # Save to HTML
    output_file = 'lda_french_visualization.html'
    pyLDAvis.save_html(vis_data, output_file)
    print(f"✓ Visualization saved to: {output_file}")

except ImportError:
    print("(pyLDAvis not installed, skipping visualization)")
except Exception as e:
    print(f"Error creating visualization: {e}")



Generating interactive visualization...
✓ Visualization saved to: lda_french_visualization.html


In [32]:
print("\n\n" + "="*70)
print("MODEL EVALUATION METRICS")
print("="*70)

# Perplexity
perplexity = lda_model.log_perplexity(corpus)
print(f"Perplexity: {perplexity:.4f}")

# Coherence
coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_docs,
    dictionary=dictionary,
    coherence='c_v'
)
coherence = coherence_model.get_coherence()
print(f"Coherence Score (C_V): {coherence:.4f}")
print(f"  Interpretation:")
print(f"    - 0.4-0.5: Weak")
print(f"    - 0.5-0.6: Moderate ← Good target")
print(f"    - 0.6-0.7: Good")
print(f"    - 0.7+: Excellent (may overfit)")



MODEL EVALUATION METRICS
Perplexity: -8.6287


/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=362533) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/lib/python3.13/multiprocessing/pope

Coherence Score (C_V): 0.2809
  Interpretation:
    - 0.4-0.5: Weak
    - 0.5-0.6: Moderate ← Good target
    - 0.6-0.7: Good
    - 0.7+: Excellent (may overfit)
